In [15]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [16]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

Q1. How many lesson pages

In [17]:
len(documents)

72

Q2. Indexing and searching

In [18]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [19]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=1
)

search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

Q3. RAG

In [20]:
from dotenv import load_dotenv
load_dotenv()

import os
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client
)

In [21]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

print(f"Input tokens: {assistant.last_response.usage_metadata.prompt_token_count}")

The agentic loop keeps calling the model until it returns a response that contains no function calls.

Specifically, the process works as follows:

1.  The loop is wrapped in a `while` statement (e.g., `while True:`).
2.  Inside the loop, the agent sends the conversation history (including previous tool results) to the LLM.
3.  The code processes the LLM's response. If the response contains a function call, the code executes the tool and appends the result to the conversation history.
4.  A flag (such as `has_function_calls`) is used to track whether any tools were requested in the current iteration.
5.  If no function calls were returned by the model, the loop breaks, ending the agentic process.

As noted in the context, the model decides when to stop. If the model returns a final answer without asking to call any more tools, the agent recognizes that the task is complete and exits the loop.
Input tokens: 7948


Q4. Chunking

In [22]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [23]:
len(chunks)

295

Q5. RAG with chunking

In [24]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

from dotenv import load_dotenv
load_dotenv()

import os
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

print(f"Input tokens: {assistant.last_response.usage_metadata.prompt_token_count}")

The agentic loop uses a `while True` loop to keep calling the model. Inside the loop, it checks the model's response for any function calls:

1. If the model returns **function calls**, the code executes them, appends the results to the message history, and sets a flag (`has_function_calls`) to `True`.
2. The loop continues to the next iteration, sending the updated message history (including the tool results) back to the model.
3. The loop stops when the model returns a response without any function calls, which triggers the `if has_function_calls == False:` condition to break the loop.

In essence, the model itself decides how many times to search, and the loop simply persists until the model provides a final answer rather than requesting further tool use.
Input tokens: 2600


Q6. Turning it into an agent (Google ADK)

In [33]:
search_tool = {
    "function_declarations": [
        {
            "name": "search",
            "description": "Search the FAQ database for entries matching the given query.",
            "parameters": {
                "type": "OBJECT",
                "properties": {
                    "query": {
                        "type": "STRING",
                        "description": "Search query text to look up in the course FAQ."
                    }
                },
                "required": ["query"]
            }
        }
    ]
}

In [34]:
import uuid
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.agents.run_config import RunConfig, StreamingMode
from google.genai import types
from google.adk.tools import FunctionTool


search_calls = 0


def search(query: str) -> dict:
    """Search the FAQ database for entries matching the given query.

    Args:
        query: Search query text to look up in the course FAQ.

    Returns:
        A dictionary with search results.
    """
    global search_calls
    search_calls += 1

    boost_dict = {"question": 3.0, "section": 0.5}
    results = index.search(query, num_results=5, boost_dict=boost_dict)
    return {"results": results}


adk_search_tool = FunctionTool(func=search)


async def agent_loop(instructions: str, question: str, model: str = "gemini-flash-lite-latest") -> str:
    global search_calls
    search_calls = 0

    agent = Agent(name="search_agent", model=model, instruction=instructions, tools=[adk_search_tool])
    runner = InMemoryRunner(agent=agent, app_name="search_app")

    session_id = str(uuid.uuid4())
    await runner.session_service.create_session(app_name="search_app", user_id="user", session_id=session_id)

    user_message = types.Content(role="user", parts=[types.Part.from_text(text=question)])

    run_config = RunConfig(streaming_mode=StreamingMode.NONE)  # <-- фікс для ADK 2.x

    last_answer = ""
    async for event in runner.run_async(
        user_id="user",
        session_id=session_id,
        new_message=user_message,
        run_config=run_config,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            last_answer = event.content.parts[0].text or ""
            break

    if not last_answer:
        raise ValueError("The agent completed the job without a text response")

    print(f"search called: {search_calls} time(s)")
    return last_answer

In [36]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()

answer = await agent_loop(instructions, "How does the agentic loop work, and how is it different from plain RAG?")
print(answer)

search called: 2 time(s)
The **agentic loop** is a design pattern where an LLM is placed in control of a workflow, enabling it to decide which actions (tools) to take, execute them, analyze the results, and determine if it has sufficient information to answer the user's question or if it needs to perform further actions.

Here is a breakdown of how it works and how it differs from plain RAG.

### How the Agentic Loop Works
The agentic loop functions as a "thought-action-observation" cycle. Instead of following a rigid, linear process, the loop repeats until the agent decides the task is complete:

1.  **Instruction/Goal:** You provide the agent with a goal (e.g., "Answer the student's question") and a set of available **tools** (e.g., a search function).
2.  **The Loop:**
    *   **Model Decision:** The agent analyzes the conversation history (which acts as its **memory**) and decides whether it has enough information to answer or if it needs to use a tool.
    *   **Action:** If it ne

Root node search_agent was cancelled.
Failed to detach context
Traceback (most recent call last):
  File "/workspaces/llm-zoomcamp-2026/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 608, in use_span
    yield span
  File "/workspaces/llm-zoomcamp-2026/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 508, in start_as_current_span
    yield span
  File "/workspaces/llm-zoomcamp-2026/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 443, in start_as_current_span
    yield span
  File "/workspaces/llm-zoomcamp-2026/.venv/lib/python3.12/site-packages/google/adk/runners.py", line 572, in _run_node_async
    yield event
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/workspaces/llm-zoomcamp-2026/.venv/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 143, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/workspaces